In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import log_loss, accuracy_score, f1_score, confusion_matrix, precision_score, recall_score, classification_report,roc_auc_score,roc_curve,RocCurveDisplay
import os
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from tqdm import tqdm

In [2]:
irrigation = pd.read_csv('train.csv', index_col = 'id')
irrigation

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,Clay,6.54,13.45,1.15,1.86,26.65,26.86,1041.33,10.62,18.85,Rice,Sowing,Kharif,Sprinkler,River,4.35,No,118.36,South,Medium
629996,Clay,7.03,54.49,0.96,2.35,36.99,88.00,1419.57,9.93,17.99,Sugarcane,Vegetative,Kharif,Drip,Groundwater,12.97,Yes,40.75,Central,Medium
629997,Clay,6.52,11.98,0.93,0.38,37.82,70.98,88.45,8.19,17.25,Potato,Vegetative,Zaid,Canal,Reservoir,13.58,Yes,2.62,South,High


In [3]:
irrigation.isna().sum()

,0
Soil_Type,0
Soil_pH,0
Soil_Moisture,0
Organic_Carbon,0
Electrical_Conductivity,0
Temperature_C,0
Humidity,0
Rainfall_mm,0
Sunlight_Hours,0
Wind_Speed_kmh,0


In [4]:
le = LabelEncoder()
irrigation["Irrigation_Need"] = le.fit_transform(irrigation["Irrigation_Need"])

In [5]:
X, y = irrigation.drop("Irrigation_Need", axis=1), irrigation["Irrigation_Need"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
ohe = OneHotEncoder(sparse_output = False, drop = 'first',handle_unknown='ignore').set_output(transform = 'pandas')
transformer = ColumnTransformer(transformers=[('OHE',ohe,make_column_selector(dtype_include=object)),
                                            ]
                               ,remainder='passthrough',
                               verbose_feature_names_out=False
                              ).set_output(transform='pandas')

In [10]:
X_train_ohe = transformer.fit_transform(X_train)
X_test_ohe = transformer.transform(X_test)

In [11]:
ss = StandardScaler()
X_train_ss = ss.fit_transform(X_train_ohe)
X_test_ss = ss.transform(X_test_ohe)

In [13]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_ss, y_train)

RandomForestClassifier(random_state=42)

In [15]:
y_pred = model.predict(X_test_ss)
accuracy_score(y_test, y_pred)

0.9849603174603174

In [16]:
X = transformer.fit_transform(X)
X = ss.fit_transform(X)
model.fit(X, y)

RandomForestClassifier(random_state=42)

In [17]:
tst = pd.read_csv("test.csv", index_col=0)
tst_ohe = transformer.transform(tst)
tst_ss = ss.transform(tst_ohe)

tst["Irrigation_Need"] = model.predict(tst_ss)
tst["Irrigation_Need"] = le.inverse_transform(tst["Irrigation_Need"])
tst

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West,Low
630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South,Low
630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East,Low
630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North,Low
630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
899995,Sandy,5.63,51.90,0.68,2.58,33.27,72.09,2326.61,7.09,10.02,Potato,Vegetative,Rabi,Rainfed,River,2.93,Yes,43.49,East,Medium
899996,Loamy,7.84,45.16,0.85,1.04,27.55,45.16,2322.37,5.15,5.62,Wheat,Vegetative,Rabi,Canal,Groundwater,11.23,Yes,92.03,West,Low
899997,Loamy,7.83,11.02,1.56,1.90,23.39,64.87,996.72,10.44,9.98,Maize,Vegetative,Zaid,Sprinkler,Groundwater,2.88,Yes,34.02,East,Medium


In [20]:
submission = pd.read_csv("sample_submission.csv")
submission["Irrigation_Need"] = le.inverse_transform(model.predict(tst_ss))
submission

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
...,...,...
269995,899995,Medium
269996,899996,Low
269997,899997,Medium
269998,899998,Low


In [21]:
submission.to_csv("kaggleSubmissionRandomForest.csv", index = False)